Loadinng image as an Array

In [1]:
from PIL import Image
import numpy as np

img = Image.open("me.jpg")    # add your own image file name here 
arr = np.array(img)
print(arr.shape)
print(arr.dtype)

""" 
o. 720 = height (rows of pixels)
o .1280 = width (columns of pixels)
o. 3 = channels (Red, Green, Blue values per pixel)
o. uint8, meaning each number is 0-255
"""


(720, 1280, 3)
uint8


' \no. 720 = height (rows of pixels)\no .1280 = width (columns of pixels)\no. 3 = channels (Red, Green, Blue values per pixel)\no. uint8, meaning each number is 0-255\n'

Grayscale

a grayscale pixel is a weighted average of R, G, and B. Human eyes are more sensitive to green than red or blue, so the standard formula weighs them differently:

gray = 0.299*R + 0.587*G + 0.114*B

In [2]:
# now here we are pull out 3 chanels using sclicing
R = arr[:, :, 0]         #(:, :, 0) means "every row, every column, channel 0"
G = arr[:, :, 1]
B = arr[:, :, 2]

gray = 0.299 * R + 0.587 * G + 0.114 * B     # here we applied formula on above 3 arrays
gray = gray.astype(np.uint8)        # result will be float bcoz we multiplied them by decimals.
                                    # so we convert them back to uint8
Image.fromarray(gray).save("gray.jpg")      # save image
print(gray.shape)

(720, 1280)


Brightness

Add a constant number (e.g. 40) to every pixel in the array.

Problem: pixel values must stay between '0-255'. Adding can push some above 255, which wraps around incorrectly in 'uint8' math. So, convert  into 'int' first, add, then use 'np.clip(array, 0, 255)' to cap it, then convert back to 'uint8'.


In [3]:
bright = arr.astype(int) + 40  #convert to int to avoid overflow, then add 40 to each pixel value
bright = np.clip(bright, 0, 255).astype(np.uint8) #clip values to be in range 0-255 and convert back to uint8
Image.fromarray(bright).save("bright.jpg")  # save image

Contrast

Pick a midpoint, '128'.
For every pixel: subtract '128', multiply by a contrast factor (try 1.5), add '128' back.
Formula: " new_pixel = (pixel - 128) * factor + 128 "
Same clipping rule applies — clip to 0-255 before converting back to uint8.

In [4]:
factor = 1.5
contrast = (arr.astype(int) - 128) * factor + 128
contrast = np.clip(contrast, 0, 255).astype(np.uint8)
Image.fromarray(contrast).save("contrast.jpg")

Flip

Horizontal flip: reverse the column order → arr[:, ::-1]

Vertical flip: reverse the row order → arr[::-1, :]

[:, ::-1] reverses columns (left-right); [::-1, :] reverses rows (top-bottom).

Think about why each slice flips in that direction based on which axis is rows vs columns.

In [5]:
flipped_h = arr[:, ::-1]
flipped_v = arr[::-1, :]
Image.fromarray(flipped_h).save("flipped_h.jpg")
Image.fromarray(flipped_v).save("flipped_v.jpg")

Crop

Pick a rectangle using row and column ranges: arr[100:300, 200:500]

First range = which rows (height) to keep, second = which columns (width) to keep.
(First slice = rows (height) you keep, second = columns (width) you keep.)

In [6]:
cropped = arr[50:600, 250:1000]         # adjust numbers to fit YOUR image's actual size
Image.fromarray(cropped).save("cropped.jpg")

Invert

For every pixel: 255 - pixel_value
Works directly on the uint8 array, no clipping needed since the result always stays in 0-255.

In [7]:
inverted = 255 - arr
Image.fromarray(inverted).save("inverted.jpg")

Blend 2 Images

Load a second image, resize it to match the first image's dimensions '(img2.resize((width, height))' using PIL before converting to array).

Convert both to arrays, then: 'blended = arr1*weight + arr2*(1-weight) (try weight=0.5).'

Clip and convert to uint8.

In [8]:
img2 = Image.open("me2.jpg")
height, width = arr.shape[0], arr.shape[1]
img2_resized = img2.resize((width, height))   
arr2 = np.array(img2_resized)

weight = 0.5
blended = arr.astype(float)*weight + arr2.astype(float)*(1-weight) 
blended = np.clip(blended, 0, 255).astype(np.uint8)
Image.fromarray(blended).save("blended.jpg")


Blur

For every pixel, average it with its neighboring pixels (left, right, up, down, and itself).

Do this using shifted slices of the array added together and divided by the count, not a loop over every pixel — that's the NumPy way.

In [9]:
arr_f = arr.astype(float)
up = np.roll(arr_f, -10, axis=0)
down = np.roll(arr_f, 10, axis=0)
left = np.roll(arr_f, -10, axis=1)
right = np.roll(arr_f, 10, axis=1)

blurred = (arr_f + up + down + left + right)/ 5
blurred = np.clip(blurred, 0, 255).astype(np.uint8)
Image.fromarray(blurred).save("blurred.jpg")